# Tag 07 — Fortgeschrittene
## Pruning and Feature Importance — Pima Indians Diabetes

تمرین‌ها:
1. `min_samples_leaf=1,5,10,20` و مقایسه Train/Test Accuracy
2. Cost-Complexity Pruning با `ccp_alpha`
3. Feature Importance به صورت bar chart

In [1]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

# Find project root automatically
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output" / "Fortgeschrittene"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)


Project root: C:\Users\esmae\Documents\Educx Kurs machine lerning\ML_Projekt_Workspace\decision-trees-predictive-modeling
Data dir: C:\Users\esmae\Documents\Educx Kurs machine lerning\ML_Projekt_Workspace\decision-trees-predictive-modeling\data
Output dir: C:\Users\esmae\Documents\Educx Kurs machine lerning\ML_Projekt_Workspace\decision-trees-predictive-modeling\output\Fortgeschrittene


In [2]:

diabetes_path = DATA_DIR / "diabetes.csv"
if not diabetes_path.exists():
    raise FileNotFoundError(f"Could not find {diabetes_path}. Put diabetes.csv into ../data/.")

df = pd.read_csv(diabetes_path)
print(df.shape)
df.head()


FileNotFoundError: Could not find C:\Users\esmae\Documents\Educx Kurs machine lerning\ML_Projekt_Workspace\decision-trees-predictive-modeling\data\diabetes.csv. Put diabetes.csv into ../data/.

In [3]:

# Expected target column for Pima dataset is usually Outcome.
target_col = "Outcome" if "Outcome" in df.columns else df.columns[-1]
print("Target column:", target_col)
print("Missing values:\n", df.isna().sum())
print("Target distribution:\n", df[target_col].value_counts(normalize=True).round(3))

X = df.drop(columns=[target_col])
y = df[target_col]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_tr.shape, X_te.shape)


NameError: name 'df' is not defined

## 1) min_samples_leaf comparison

In [4]:

leaves = [1, 5, 10, 20]
rows = []
for leaf in leaves:
    model = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=42)
    model.fit(X_tr, y_tr)
    rows.append({
        "min_samples_leaf": leaf,
        "train_accuracy": model.score(X_tr, y_tr),
        "test_accuracy": model.score(X_te, y_te),
        "tree_depth": model.tree_.max_depth,
        "node_count": model.tree_.node_count
    })

leaf_df = pd.DataFrame(rows)
leaf_df.to_csv(OUTPUT_DIR / "intermediate_min_samples_leaf_results.csv", index=False)
leaf_df


NameError: name 'X_tr' is not defined

In [5]:

plt.figure(figsize=(8,5))
plt.plot(leaf_df["min_samples_leaf"], leaf_df["train_accuracy"], marker="o", label="Train")
plt.plot(leaf_df["min_samples_leaf"], leaf_df["test_accuracy"], marker="o", label="Test")
plt.xlabel("min_samples_leaf")
plt.ylabel("Accuracy")
plt.title("Train vs Test Accuracy by min_samples_leaf")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "intermediate_min_samples_leaf_accuracy.png", dpi=150)
plt.show()


NameError: name 'leaf_df' is not defined

<Figure size 800x500 with 0 Axes>

## 2) Cost-Complexity Pruning

In [6]:
from sklearn.tree import DecisionTreeClassifier

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_tr = X_tr.reset_index(drop=True)
y_tr = y_tr.reset_index(drop=True)

model = DecisionTreeClassifier(
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_tr, y_tr)

y_pred = model.predict(X_te)

acc = accuracy_score(y_te, y_pred)
print(f"Test Accuracy: {acc:.4f}")

cm = confusion_matrix(y_te, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=model.classes_
)

disp.plot()
plt.title("Confusion Matrix — Decision Tree with min_samples_leaf=5")
plt.show()


NameError: name 'X_tr' is not defined

In [8]:

base_tree = DecisionTreeClassifier(random_state=42)
path = base_tree.cost_complexity_pruning_path(X_tr, y_tr)
alphas = path.ccp_alphas

# Often the last alpha collapses the tree to root only; keep it but mark it.
rows = []
for alpha in alphas:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    model.fit(X_tr, y_tr)
    rows.append({
        "ccp_alpha": alpha,
        "train_accuracy": model.score(X_tr, y_tr),
        "test_accuracy": model.score(X_te, y_te),
        "tree_depth": model.tree_.max_depth,
        "node_count": model.tree_.node_count
    })

alpha_df = pd.DataFrame(rows)
alpha_df.to_csv(OUTPUT_DIR / "intermediate_ccp_alpha_results.csv", index=False)
best_row = alpha_df.loc[alpha_df["test_accuracy"].idxmax()]
print("Best alpha row:")
print(best_row)
alpha_df.head()


NameError: name 'X_tr' is not defined

In [9]:

plt.figure(figsize=(9,5))
plt.plot(alpha_df["ccp_alpha"], alpha_df["test_accuracy"], marker="o", label="Test")
plt.plot(alpha_df["ccp_alpha"], alpha_df["train_accuracy"], marker="o", label="Train")
plt.xlabel("ccp_alpha")
plt.ylabel("Accuracy")
plt.title("Cost-Complexity Pruning: Accuracy vs ccp_alpha")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "intermediate_ccp_alpha_accuracy.png", dpi=150)
plt.show()


NameError: name 'alpha_df' is not defined

<Figure size 900x500 with 0 Axes>

## 3) Feature Importance

In [10]:

best_alpha = float(best_row["ccp_alpha"])
dt_best = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
dt_best.fit(X_tr, y_tr)

fi = pd.Series(dt_best.feature_importances_, index=X.columns).sort_values()
fi.to_csv(OUTPUT_DIR / "intermediate_feature_importances.csv", header=["importance"])

plt.figure(figsize=(8,5))
fi.plot(kind="barh")
plt.xlabel("Importance")
plt.title("Decision Tree Feature Importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "intermediate_feature_importance.png", dpi=150)
plt.show()

fi.sort_values(ascending=False)


NameError: name 'best_row' is not defined

## What to send back

- `intermediate_min_samples_leaf_results.csv`
- `intermediate_ccp_alpha_results.csv`
- `intermediate_feature_importances.csv`
- plots from `output/Fortgeschrittene/`